# handlers

> Handler wrappers for cross-domain coordination (alignment status updates)

In [ ]:
#| default_exp components.handlers

In [ ]:
#| export
import asyncio
from functools import wraps
from typing import Callable, Any, List, Optional

from fasthtml.common import Div, Span, Button

from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_fasthtml_card_stack.components.controls import render_width_slider
from cjm_fasthtml_card_stack.core.constants import DEFAULT_VISIBLE_COUNT
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_styles, badge_sizes
from cjm_fasthtml_tailwind.utilities.layout import display_tw
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_transcript_segment_align.html_ids import CombinedHtmlIds
from cjm_transcript_segment_align.components.step_renderer import (
    render_alignment_status, render_seg_mini_stats_badge, render_align_mini_stats_badge,
    render_footer_inner_content,
)
from cjm_transcript_segment_align.components.keyboard_config import (
    build_combined_kb_system, render_keyboard_hints_collapsible,
    generate_zone_change_js, SWITCH_CHROME_BTN_ID,
)
from cjm_transcript_segmentation.models import TextSegment, SegmentationUrls
from cjm_transcript_segmentation.routes.core import WorkflowStateStore
from cjm_transcript_segmentation.routes.handlers import (
    SegInitResult, SegMutationResult, build_mutation_response,
    _handle_seg_init,
    _handle_seg_split_result, _handle_seg_merge_result,
    _handle_seg_undo_result, _handle_seg_reset_result,
    _handle_seg_ai_split_result,
)
from cjm_transcript_segmentation.components.step_renderer import (
    render_toolbar, render_seg_footer_content,
)
from cjm_transcript_segmentation.components.card_stack_config import (
    SEG_CS_CONFIG, SEG_CS_IDS,
)
from cjm_transcript_vad_align.models import AlignmentUrls, VADChunk
from cjm_transcript_vad_align.routes.core import (
    WorkflowStateStore as AlignWorkflowStateStore,
)
from cjm_transcript_vad_align.routes.handlers import (
    AlignInitResult, _handle_align_init,
)
from cjm_transcript_vad_align.services.alignment import AlignmentService
from cjm_transcript_source_select.services.source import SourceService

from cjm_transcript_segment_align.routes.forced_alignment import (
    render_fa_trigger_button, render_fa_toggle,
)

## FA Extra Actions Helper

Builds the FA controls content for the toolbar `extra_actions` slot based on current state.

In [ ]:
#| export
def _find_session_id(args, kwargs):
    """Find session_id from args or kwargs."""
    if 'sess' in kwargs:
        try:
            return get_session_id(kwargs['sess'])
        except:
            pass
    for arg in args:
        try:
            session_id = get_session_id(arg)
            if session_id:
                return session_id
        except:
            continue
    return None


def segments_match_presplit(
    current_segments: list,  # Current segment dicts
    presplit: list,  # Pre-split snapshot segment dicts
) -> bool:  # Whether current segments match the pre-split
    """Check if current segments match a pre-split snapshot by text content."""
    if len(current_segments) != len(presplit):
        return False
    return all(
        c.get("text", "") == p.get("text", "")
        for c, p in zip(current_segments, presplit)
    )


def build_fa_extra_actions(
    seg_state: dict,  # Segmentation step state dict
    fa_trigger_url: str = "",  # URL for FA trigger route
    fa_toggle_url: str = "",  # URL for FA toggle route
    fa_available: bool = False,  # Whether FA plugin is available
) -> Any:  # FA controls wrapped in slot div (or None)
    """Build FA controls for the toolbar extra_actions slot.
    
    Shows toggle when current segments match either pre-split snapshot.
    Shows trigger button when FA pre-split doesn't exist or current doesn't match either.
    """
    if not fa_available:
        return None

    fa_presplit = seg_state.get("fa_presplit")
    if fa_presplit is None:
        # No FA pre-split exists — show trigger button
        return Div(render_fa_trigger_button(fa_trigger_url), id=CombinedHtmlIds.FA_TOOLBAR_SLOT)

    # FA pre-split exists — check if current matches either snapshot
    current = seg_state.get("segments", [])
    nltk = seg_state.get("nltk_presplit", [])

    if segments_match_presplit(current, nltk):
        return Div(render_fa_toggle("nltk", fa_toggle_url), id=CombinedHtmlIds.FA_TOOLBAR_SLOT)
    elif segments_match_presplit(current, fa_presplit):
        return Div(render_fa_toggle("forced_alignment", fa_toggle_url), id=CombinedHtmlIds.FA_TOOLBAR_SLOT)
    else:
        # Current doesn't match either — show trigger button (can re-run FA)
        return Div(render_fa_trigger_button(fa_trigger_url), id=CombinedHtmlIds.FA_TOOLBAR_SLOT)


def create_seg_mutation_wrapper(
    handler_result: Callable,  # _handle_seg_*_result function
    fa_trigger_url: str = "",  # URL for FA trigger route
    fa_toggle_url: str = "",  # URL for FA toggle route
    fa_available: bool = False,  # Whether FA plugin is available
    clear_fa_presplit: bool = False,  # Whether to clear fa_presplit after handler (for NLTK Split)
) -> Callable:  # Wrapped handler that builds OOB with FA extra_actions + alignment status
    """Create a wrapped mutation handler that uses SegMutationResult.
    
    Calls the _result handler variant, builds targeted OOB response with FA
    controls in toolbar, and appends alignment status + mini-stats OOB.
    Computes nltk_split_disabled from state for toolbar rendering.
    
    When clear_fa_presplit=True (used for NLTK Split), clears the FA pre-split
    snapshot so the toggle is replaced with the Force Align button.
    """
    @wraps(handler_result)
    async def wrapped(
        state_store: WorkflowStateStore,
        workflow_id: str,
        *args,
        **kwargs
    ):
        # Call the _result handler (returns SegMutationResult)
        result = handler_result(state_store, workflow_id, *args, **kwargs)
        if asyncio.iscoroutine(result):
            result = await result

        # Find session_id for cross-domain state reads
        session_id = _find_session_id(args, kwargs)

        # Build FA extra_actions and nltk_split_disabled from state
        fa_extra = None
        nltk_disabled = False
        alignment_status_oob = ()
        mini_stats_oob = ()

        if session_id is not None:
            workflow_state = state_store.get_state(workflow_id, session_id)
            step_states = workflow_state.get("step_states", {})
            seg_state = step_states.get("segmentation", {})

            # Clear FA pre-split if requested (NLTK Split discards FA snapshot)
            if clear_fa_presplit and "fa_presplit" in seg_state:
                del seg_state["fa_presplit"]
                step_states["segmentation"] = seg_state
                workflow_state["step_states"] = step_states
                state_store.update_state(workflow_id, session_id, workflow_state)

            chunk_count = len(step_states.get("alignment", {}).get("vad_chunks", []))
            segment_count = len(result.segment_dicts)

            fa_extra = build_fa_extra_actions(
                seg_state, fa_trigger_url, fa_toggle_url, fa_available,
            )

            # Check if current matches NLTK pre-split for button disabled state
            nltk_presplit = seg_state.get("nltk_presplit", [])
            nltk_disabled = segments_match_presplit(result.segment_dicts, nltk_presplit)

            alignment_status_oob = (render_alignment_status(segment_count, chunk_count, oob=True),)
            segments = [TextSegment.from_dict(d) for d in result.segment_dicts]
            mini_stats_oob = (render_seg_mini_stats_badge(segments, oob=True),)

        # Build targeted OOB response with FA controls in toolbar
        urls = kwargs.get("urls")
        seg_oob = build_mutation_response(
            result.segment_dicts, result.focused_index, result.visible_count,
            result.history_depth, urls, is_auto_mode=result.is_auto_mode,
            extra_actions=fa_extra, nltk_split_disabled=nltk_disabled,
        )

        return (*seg_oob, *result.extra_oob, *alignment_status_oob, *mini_stats_oob)

    return wrapped

In [ ]:
#| export
def wrap_align_mutation_handler(
    handler: Callable,  # Handler function to wrap
) -> Callable:  # Wrapped handler that appends alignment status OOB
    """Wrap an alignment mutation handler to add alignment status OOB.
    
    The handler is expected to take (state_store, workflow_id, ...) as first params.
    """
    @wraps(handler)
    async def wrapped(
        state_store: WorkflowStateStore,
        workflow_id: str,
        *args,
        **kwargs
    ):
        if asyncio.iscoroutinefunction(handler):
            result = await handler(state_store, workflow_id, *args, **kwargs)
        else:
            result = handler(state_store, workflow_id, *args, **kwargs)
        
        session_id = _find_session_id(args, kwargs)
        
        if session_id is not None:
            workflow_state = state_store.get_state(workflow_id, session_id)
            step_states = workflow_state.get("step_states", {})
            segment_count = len(step_states.get("segmentation", {}).get("segments", []))
            chunk_count = len(step_states.get("alignment", {}).get("vad_chunks", []))
            
            return (*result, render_alignment_status(segment_count, chunk_count, oob=True))
        
        return result
    
    return wrapped

## Init Chrome Wrappers

Wrappers for init handlers that build the full combined KB system
and populate shared chrome containers. These are more complex than
mutation wrappers because they need to set up the keyboard system
and all shared chrome elements on first load.

In [ ]:
#| export
def create_seg_init_chrome_wrapper(
    align_urls:AlignmentUrls,  # URL bundle for alignment routes (for KB system)
    switch_chrome_url:str,  # URL for chrome switching (for KB system)
    fa_trigger_url:str="",  # URL for forced alignment trigger (optional)
    fa_toggle_url:str="",  # URL for forced alignment toggle (optional)
    fa_available:bool=False,  # Whether forced alignment plugin is available
) -> Callable:  # Wrapped handler that builds KB system and shared chrome
    """Create a wrapper for seg init that builds combined KB system and shared chrome.
    
    Saves nltk_presplit snapshot at init time for match detection.
    FA controls are rendered in the toolbar via extra_actions.
    """
    async def wrapped_seg_init(
        state_store:WorkflowStateStore,
        workflow_id:str,
        source_service:SourceService,
        segmentation_service:Any,
        request:Any,
        sess:Any,
        urls:SegmentationUrls,
        visible_count:int=5,
        card_width:int=40,
    ):
        """Wrapped seg init that adds KB system and shared chrome."""
        # Call pure domain handler
        result: SegInitResult = await _handle_seg_init(
            state_store, workflow_id, source_service, segmentation_service,
            request, sess, urls, visible_count, card_width,
        )
        
        session_id = get_session_id(sess)
        
        # Save NLTK pre-split snapshot for match detection
        workflow_state = state_store.get_state(workflow_id, session_id)
        step_states = workflow_state.get("step_states", {})
        seg_state = step_states.get("segmentation", {})

        # Only save nltk_presplit if not already saved (avoid overwriting on re-init)
        if "nltk_presplit" not in seg_state:
            seg_state["nltk_presplit"] = seg_state.get("segments", [])[:]
            step_states["segmentation"] = seg_state
            workflow_state["step_states"] = step_states
            state_store.update_state(workflow_id, session_id, workflow_state)

        chunk_count = len(step_states.get("alignment", {}).get("vad_chunks", []))
        segment_count = len(result.segments)
        
        # Build combined KB system with both zones
        kb_manager, kb_system = build_combined_kb_system(urls, align_urls)
        
        # OOB swap for stable keyboard system container
        kb_system_oob = Div(
            kb_system.script,
            kb_system.hidden_inputs,
            kb_system.action_buttons,
            id=CombinedHtmlIds.KEYBOARD_SYSTEM,
            hx_swap_oob="innerHTML"
        )
        
        # Zone change JS (goes in response for browser to execute)
        zone_change_js = generate_zone_change_js(switch_chrome_url)
        
        # Hidden chrome switch button for HTMX trigger
        chrome_switch_btn = Button(
            id=SWITCH_CHROME_BTN_ID,
            cls=str(display_tw.hidden),
            hx_post=switch_chrome_url,
            hx_include=f"#{CombinedHtmlIds.ACTIVE_COLUMN_INPUT}",
            hx_swap="none",
            hx_swap_oob="true",
        )
        
        # Update hints to include zone switch info
        hints_oob = Div(
            render_keyboard_hints_collapsible(kb_manager, include_zone_switch=True),
            id=CombinedHtmlIds.SHARED_HINTS,
            hx_swap_oob="innerHTML"
        )
        
        # Build FA extra_actions for toolbar
        fa_extra = build_fa_extra_actions(
            seg_state, fa_trigger_url, fa_toggle_url, fa_available,
        )
        
        # Toolbar OOB (with FA controls in extra_actions, NLTK Split disabled at init)
        toolbar_oob = Div(
            render_toolbar(
                reset_url=urls.reset, ai_split_url=urls.ai_split, undo_url=urls.undo,
                can_undo=(result.history_depth > 0),
                visible_count=result.visible_count,
                is_auto_mode=result.is_auto_mode,
                extra_actions=fa_extra,
                nltk_split_disabled=True,
            ),
            id=CombinedHtmlIds.SHARED_TOOLBAR,
            hx_swap_oob="innerHTML"
        )
        
        # Controls OOB (width slider)
        controls_oob = Div(
            render_width_slider(SEG_CS_CONFIG, SEG_CS_IDS, card_width=result.card_width),
            id=CombinedHtmlIds.SHARED_CONTROLS,
            hx_swap_oob="innerHTML"
        )
        
        # Footer OOB with alignment status
        footer_oob = Div(
            render_footer_inner_content(
                render_seg_footer_content(result.segments, result.focused_index),
                segment_count, chunk_count
            ),
            id=CombinedHtmlIds.SHARED_FOOTER,
            hx_swap_oob="innerHTML"
        )
        
        # Mini-stats badge OOB
        mini_stats_oob = render_seg_mini_stats_badge(result.segments, oob=True)
        
        return (
            result.column_body, kb_system_oob, zone_change_js, chrome_switch_btn,
            hints_oob, toolbar_oob, controls_oob, footer_oob, mini_stats_oob,
        )
    
    return wrapped_seg_init

In [ ]:
#| export
def create_align_init_chrome_wrapper() -> Callable:  # Wrapped handler that adds alignment status
    """Create a wrapper for align init that adds mini-stats and alignment status.
    
    Returns a footer OOB (not a standalone alignment status badge) to avoid
    a race condition: both seg and align init auto-trigger on load, and the
    alignment status badge only exists inside the footer after seg init's
    footer OOB is processed. Using a footer OOB is safe because the footer
    container always exists in the DOM.
    """
    async def wrapped_align_init(
        state_store:WorkflowStateStore,
        workflow_id:str,
        source_service:SourceService,
        alignment_service:AlignmentService,
        request:Any,
        sess:Any,
        urls:AlignmentUrls,
        visible_count:int=5,
        card_width:int=40,
    ):
        """Wrapped align init that adds mini-stats and alignment status in footer."""
        from cjm_transcript_segmentation.components.step_renderer import render_seg_footer_content
        from cjm_fasthtml_tailwind.utilities.typography import font_size as fs
        from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui as td

        # Call pure domain handler
        result: AlignInitResult = await _handle_align_init(
            state_store, workflow_id, source_service, alignment_service,
            request, sess, urls, visible_count, card_width,
        )
        
        session_id = get_session_id(sess)
        
        # Get segment state for footer rendering and alignment status
        workflow_state = state_store.get_state(workflow_id, session_id)
        seg_state = workflow_state.get("step_states", {}).get("segmentation", {})
        segment_count = len(seg_state.get("segments", []))
        chunk_count = len(result.chunks)
        
        # Mini-stats badge OOB
        mini_stats_oob = render_align_mini_stats_badge(result.chunks, oob=True)
        
        # Footer OOB with alignment status (targets sd-shared-footer which always exists,
        # unlike sd-alignment-status which is inside the footer and may not exist yet)
        segments = [TextSegment.from_dict(s) for s in seg_state.get("segments", [])]
        focused_index = seg_state.get("focused_index", 0)
        
        if segments:
            column_footer = render_seg_footer_content(segments, focused_index)
        else:
            column_footer = Span(
                "Footer with progress and timestamp details will appear here.",
                cls=combine_classes(fs.sm, td.base_content.opacity(50))
            )
        
        footer_oob = Div(
            render_footer_inner_content(column_footer, segment_count, chunk_count),
            id=CombinedHtmlIds.SHARED_FOOTER,
            hx_swap_oob="innerHTML"
        )
        
        return (result.column_body, mini_stats_oob, footer_oob)
    
    return wrapped_align_init

## Mutation Wrapper Factory

Creates wrapped mutation handlers with FA controls in toolbar.
Called at setup time when FA URLs are known.

In [ ]:
#| export
def create_seg_mutation_wrappers(
    fa_trigger_url: str = "",  # URL for FA trigger route
    fa_toggle_url: str = "",  # URL for FA toggle route
    fa_available: bool = False,  # Whether FA plugin is available
) -> dict:  # Dict with keys: split, merge, undo, reset, ai_split
    """Create wrapped mutation handlers with FA controls in toolbar.
    
    Returns a dict of handler name → wrapped handler function.
    Called at setup time when FA URLs are known.
    The ai_split wrapper has clear_fa_presplit=True — clicking NLTK Split
    discards the FA pre-split snapshot and replaces the toggle with the trigger button.
    """
    fa_kwargs = dict(
        fa_trigger_url=fa_trigger_url,
        fa_toggle_url=fa_toggle_url,
        fa_available=fa_available,
    )
    return {
        "split": create_seg_mutation_wrapper(_handle_seg_split_result, **fa_kwargs),
        "merge": create_seg_mutation_wrapper(_handle_seg_merge_result, **fa_kwargs),
        "undo": create_seg_mutation_wrapper(_handle_seg_undo_result, **fa_kwargs),
        "reset": create_seg_mutation_wrapper(_handle_seg_reset_result, **fa_kwargs),
        "ai_split": create_seg_mutation_wrapper(
            _handle_seg_ai_split_result, **fa_kwargs, clear_fa_presplit=True,
        ),
    }

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()